# First ideas file_server_clean_up — Safe file audit & cleanup candidates

**What this notebook does**
- Recursively scans a target folder
- Flags obvious junk (OS temp/metadata, zero-byte files, etc.)
- Detects *content duplicates* using hashes (safe, filename-independent)
- Chooses one file to **KEEP** per duplicate set (configurable)
- Exports a **CSV** listing paths and files flagged as **DELETE_CANDIDATE** (no deletion by default)

**Safety**: This notebook does **not** delete anything unless you explicitly enable the optional quarantine step at the end.

## 0) Install checklist (run once)
You should already have created the conda environment and installed dependencies. See the repo README / setup steps from the chat.

In [1]:
import os
import sys
import math
import time
import hashlib
from dataclasses import dataclass
from pathlib import Path
from datetime import datetime

import pandas as pd
from tqdm import tqdm

print('Python:', sys.version)
print('CWD:', os.getcwd())

Python: 3.12.12 | packaged by Anaconda, Inc. | (main, Oct 21 2025, 20:05:38) [MSC v.1929 64 bit (AMD64)]
CWD: c:\00_dev\file_server_clean_up\notebooks


## 1) Configuration (edit these)


In [2]:
# === REQUIRED: set the folder you want to scan ===
# Windows example:
# ROOT = Path(r"D:\\FILE_SERVER_EXPORT")
# Network share example:
# ROOT = Path(r"\\\\SERVER\\Share\\SomeFolder")

ROOT = Path(r"Z:\\SONADO-IK-801455240\\10_PROJECTS\\HTL0049-01_OITYLO-KOKKALA_MANI\\04_DESIGN_ENGINEERING")

# Where to write outputs
OUTPUT_DIR = Path("..") / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Exclusions
EXCLUDE_DIR_NAMES = {
    ".git", ".ipynb_checkpoints", "__pycache__", "node_modules",
    "$RECYCLE.BIN", "System Volume Information"
}

# Obvious junk filenames/extensions (add as needed)
JUNK_FILENAMES = {
    ".DS_Store", "Thumbs.db", "desktop.ini", "Icon\r", "._.DS_Store"
}
JUNK_EXTENSIONS = {
    ".tmp", ".temp", ".swp", ".swo", ".bak", ".old", ".dwl", ".dwl2", ".~lock"
}

# Duplicate detection / hashing
HASH_ALGO = "blake2b"   # blake2b is fast & in hashlib
FAST_HASH_BYTES = 2 * 1024 * 1024  # 2 MiB from head + 2 MiB from tail
FULL_HASH_FOR_FINAL_GROUPS = True  # set False to speed up but risk rare collisions

# How to pick the KEEP file within each duplicate group
# Options: "newest_mtime", "oldest_mtime", "shortest_path"
DUP_KEEP_STRATEGY = "newest_mtime"

# Optional directory preferences (higher wins). Example: prefer a 'CURRENT' folder.
# Provide as substrings (case-insensitive) or absolute paths.
PREFERRED_PATH_SUBSTRINGS = [
    # r"\\CURRENT\\",
    # r"\\_LATEST\\",
]

# Minimum file size (bytes) to consider for duplicate grouping
MIN_SIZE_FOR_DUP_CHECK = 1

# CSV name
RUN_TAG = datetime.now().strftime("%Y%m%d_%H%M%S")
CSV_PATH = OUTPUT_DIR / f"cleanup_candidates_{RUN_TAG}.csv"

assert ROOT.exists(), f"ROOT does not exist: {ROOT}"

## 2) File scan (metadata only)


In [3]:
def iter_files(root: Path):
    # Conservative walk: ignore symlinks by default
    for dirpath, dirnames, filenames in os.walk(root):
        # prune excluded directories in-place
        dirnames[:] = [d for d in dirnames if d not in EXCLUDE_DIR_NAMES]
        for fn in filenames:
            p = Path(dirpath) / fn
            try:
                if p.is_symlink():
                    continue
                st = p.stat()
            except (FileNotFoundError, PermissionError, OSError):
                continue
            yield {
                "path": str(p),
                "name": p.name,
                "ext": p.suffix.lower(),
                "size": int(st.st_size),
                "mtime": float(st.st_mtime),
            }

rows = []
for rec in tqdm(iter_files(ROOT), desc="Scanning", unit="files"):
    rows.append(rec)

df = pd.DataFrame(rows)
df["mtime_iso"] = pd.to_datetime(df["mtime"], unit="s", utc=True).dt.tz_convert(None).astype(str)
df["is_zero_bytes"] = df["size"] == 0
df["is_junk_name"] = df["name"].isin(JUNK_FILENAMES)
df["is_junk_ext"] = df["ext"].isin(JUNK_EXTENSIONS)
df["junk_reason"] = ""
df.loc[df["is_zero_bytes"], "junk_reason"] += "zero_bytes;"
df.loc[df["is_junk_name"], "junk_reason"] += "junk_filename;"
df.loc[df["is_junk_ext"], "junk_reason"] += "junk_extension;"

df.shape, df.head()

Scanning: 440files [01:49,  4.02files/s]


((440, 10),
                                                 path  \
 0  Z:\SONADO-IK-801455240\10_PROJECTS\HTL0049-01_...   
 1  Z:\SONADO-IK-801455240\10_PROJECTS\HTL0049-01_...   
 2  Z:\SONADO-IK-801455240\10_PROJECTS\HTL0049-01_...   
 3  Z:\SONADO-IK-801455240\10_PROJECTS\HTL0049-01_...   
 4  Z:\SONADO-IK-801455240\10_PROJECTS\HTL0049-01_...   
 
                                         name   ext     size         mtime  \
 0  SONADO_OITYLO_TOMH TOIXOY KAI TOIXIOY.dwg  .dwg   621200  1.667403e+09   
 1                                  Thumbs.db   .db     9216  1.772458e+09   
 2                06_SONADO_OITYLO_OPSEIS.pdf  .pdf  4809161  1.650031e+09   
 3                 07_SONADO_OITYLO_TOMES.pdf  .pdf  3763026  1.650031e+09   
 4   08_SONADO_OITYLO_DIAGRAMMA AKALYPTOY.pdf  .pdf  8410296  1.650031e+09   
 
                        mtime_iso  is_zero_bytes  is_junk_name  is_junk_ext  \
 0  2022-11-02 15:31:59.000000000          False         False        False   
 1  2026-03-02 1

## 3) Hashing helpers


In [4]:
def _new_hasher(algo: str):
    if algo == "blake2b":
        return hashlib.blake2b(digest_size=32)
    if algo == "sha256":
        return hashlib.sha256()
    raise ValueError(f"Unsupported algo: {algo}")

def fast_fingerprint(path: str, size: int, algo: str = HASH_ALGO, nbytes: int = FAST_HASH_BYTES) -> str:
    """Fast fingerprint: hash(first nbytes + last nbytes + size)."""
    h = _new_hasher(algo)
    h.update(str(size).encode("utf-8"))
    p = Path(path)
    with p.open("rb") as f:
        head = f.read(nbytes)
        h.update(head)
        if size > nbytes:
            # tail
            try:
                f.seek(max(0, size - nbytes))
                tail = f.read(nbytes)
                h.update(tail)
            except OSError:
                # some special files may not seek cleanly
                pass
    return h.hexdigest()

def full_hash(path: str, algo: str = HASH_ALGO, chunk_size: int = 8 * 1024 * 1024) -> str:
    h = _new_hasher(algo)
    p = Path(path)
    with p.open("rb") as f:
        while True:
            b = f.read(chunk_size)
            if not b:
                break
            h.update(b)
    return h.hexdigest()

## 4) Duplicate detection (size → fast hash → full hash)


In [5]:
df["dup_eligible"] = df["size"] >= MIN_SIZE_FOR_DUP_CHECK

# Pass 1: only consider size groups with >1 files
size_counts = df.loc[df["dup_eligible"], "size"].value_counts()
candidate_sizes = set(size_counts[size_counts > 1].index.tolist())

cand = df[df["dup_eligible"] & df["size"].isin(candidate_sizes)].copy()
print("Duplicate candidates by size:", len(cand), "files across", len(candidate_sizes), "sizes")

# Pass 2: compute fast fingerprints for candidates
fast_hashes = []
for path, size in tqdm(zip(cand["path"], cand["size"]), total=len(cand), desc="Fast hashing"):
    try:
        fast_hashes.append(fast_fingerprint(path, int(size)))
    except (FileNotFoundError, PermissionError, OSError):
        fast_hashes.append(None)

cand["hash_fast"] = fast_hashes
cand = cand.dropna(subset=["hash_fast"]).copy()

# Pass 3: group by (size, hash_fast), and optionally full-hash within groups >1
cand["group_fast"] = cand.groupby(["size", "hash_fast"]).ngroup()
fast_group_sizes = cand["group_fast"].value_counts()
fast_dupe_groups = fast_group_sizes[fast_group_sizes > 1].index.tolist()

cand["hash_full"] = None
if FULL_HASH_FOR_FINAL_GROUPS and fast_dupe_groups:
    mask = cand["group_fast"].isin(fast_dupe_groups)
    sub = cand.loc[mask].copy()
    fulls = []
    for path in tqdm(sub["path"], total=len(sub), desc="Full hashing"):
        try:
            fulls.append(full_hash(path))
        except (FileNotFoundError, PermissionError, OSError):
            fulls.append(None)
    cand.loc[mask, "hash_full"] = fulls

# Determine the final duplicate key
cand["dup_key"] = cand.apply(
    lambda r: (r["size"], r["hash_full"]) if (FULL_HASH_FOR_FINAL_GROUPS and pd.notna(r["hash_full"])) else (r["size"], r["hash_fast"]),
    axis=1,
)

cand["dup_group_id"] = cand.groupby("dup_key").ngroup()
dup_group_counts = cand["dup_group_id"].value_counts()
dup_groups = dup_group_counts[dup_group_counts > 1].index.tolist()

print("Final duplicate groups:", len(dup_groups))
dup_group_counts.head(10)

Duplicate candidates by size: 842 files across 340 sizes


Full hashing: 100%|██████████| 830/830 [06:56<00:00,  1.99it/s]

Final duplicate groups: 339


dup_group_id
100    4
103    4
332    4
333    4
347    4
346    4
337    4
338    4
176    4
340    4
Name: count, dtype: int64

## 5) Decide KEEP vs DELETE_CANDIDATE


In [6]:
def preference_score(path: str) -> int:
    p = path.lower()
    score = 0
    for i, sub in enumerate(PREFERRED_PATH_SUBSTRINGS):
        if sub and sub.lower() in p:
            # earlier in list = higher weight
            score += (len(PREFERRED_PATH_SUBSTRINGS) - i) * 10_000
    # shorter path gets a tiny bump (helps tie-break)
    score += max(0, 500 - len(path))
    return score

cand["pref_score"] = cand["path"].apply(preference_score)

def pick_keeper(group: pd.DataFrame) -> int:
    if DUP_KEEP_STRATEGY == "newest_mtime":
        # preference score dominates, then newest mtime
        g = group.sort_values(["pref_score", "mtime"], ascending=[False, False])
    elif DUP_KEEP_STRATEGY == "oldest_mtime":
        g = group.sort_values(["pref_score", "mtime"], ascending=[False, True])
    elif DUP_KEEP_STRATEGY == "shortest_path":
        g = group.assign(path_len=group["path"].str.len()).sort_values(["pref_score", "path_len"], ascending=[False, True])
    else:
        raise ValueError(f"Unknown strategy: {DUP_KEEP_STRATEGY}")
    return g.index[0]

# Mark keepers within each duplicate group
cand["dup_is_keeper"] = False
keepers = []
for gid, g in tqdm(cand[cand["dup_group_id"].isin(dup_groups)].groupby("dup_group_id"), desc="Choosing keepers"):
    keep_idx = pick_keeper(g)
    keepers.append(keep_idx)
cand.loc[keepers, "dup_is_keeper"] = True

# Merge duplicate info back to full df
df = df.merge(
    cand[["path", "hash_fast", "hash_full", "dup_group_id", "dup_is_keeper"]],
    on="path", how="left"
)

# Decision logic (conservative):
# - If junk => DELETE_CANDIDATE
# - Else if in duplicate group and not keeper => DELETE_CANDIDATE
# - Else => KEEP
df["decision"] = "KEEP"
df["reason"] = "";
df.loc[df["junk_reason"] != "", "decision"] = "DELETE_CANDIDATE"
df.loc[df["junk_reason"] != "", "reason"] += df.loc[df["junk_reason"] != "", "junk_reason"]

dup_nonkeepers = df["dup_group_id"].notna() & (df["dup_is_keeper"] == False)
df.loc[dup_nonkeepers & (df["decision"] == "KEEP"), "decision"] = "DELETE_CANDIDATE"
df.loc[dup_nonkeepers & (df["decision"] == "DELETE_CANDIDATE"), "reason"] += "duplicate_nonkeeper;"
df.loc[dup_nonkeepers & (df["decision"] == "KEEP"), "reason"] += "duplicate_nonkeeper;"

# For keepers in a dupe group, note it
df.loc[df["dup_group_id"].notna() & (df["dup_is_keeper"] == True), "reason"] += "duplicate_keeper;"

# Clean reason formatting
df["reason"] = df["reason"].str.strip(";")

df["decision"].value_counts(), df.head()

Choosing keepers: 100%|██████████| 339/339 [00:00<00:00, 3324.35it/s]


(decision
 DELETE_CANDIDATE    527
 KEEP                488
 Name: count, dtype: int64,
                                                 path  \
 0  Z:\SONADO-IK-801455240\10_PROJECTS\HTL0049-01_...   
 1  Z:\SONADO-IK-801455240\10_PROJECTS\HTL0049-01_...   
 2  Z:\SONADO-IK-801455240\10_PROJECTS\HTL0049-01_...   
 3  Z:\SONADO-IK-801455240\10_PROJECTS\HTL0049-01_...   
 4  Z:\SONADO-IK-801455240\10_PROJECTS\HTL0049-01_...   
 
                                         name   ext     size         mtime  \
 0  SONADO_OITYLO_TOMH TOIXOY KAI TOIXIOY.dwg  .dwg   621200  1.667403e+09   
 1         01_SONADO_OITYLO_KATOPSH ST A'.pdf  .pdf  1865245  1.650031e+09   
 2         02_SONADO_OITYLO_KATOPSH ST B'.pdf  .pdf  4482305  1.650031e+09   
 3         03_SONADO_OITYLO_KATOPSH ST C'.pdf  .pdf  5285680  1.650031e+09   
 4         04_SONADO_OITYLO_KATOPSH ST D'.pdf  .pdf  4610105  1.650031e+09   
 
                        mtime_iso  is_zero_bytes  is_junk_name  is_junk_ext  \
 0  2022-11-02 15:3

## 6) Export CSV (paths + flags)


In [7]:
out = df[[
    "path", "decision", "reason", "size", "mtime_iso",
    "ext", "hash_fast", "hash_full", "dup_group_id", "dup_is_keeper"
]].copy()

out.to_csv(CSV_PATH, index=False, encoding="utf-8")
print("Wrote:", CSV_PATH)
out["decision"].value_counts()

Wrote: ..\outputs\cleanup_candidates_20260302_141826.csv


decision
DELETE_CANDIDATE    527
KEEP                488
Name: count, dtype: int64

## 7) Review helpers


In [8]:
# Preview what would be deleted
out[out["decision"] == "DELETE_CANDIDATE"].head(50)

,path,decision,reason,size,mtime_iso,ext,hash_fast,hash_full,dup_group_id,dup_is_keeper
35,Z:\SONADO-IK-801455240\10_PROJECTS\HTL0049-01_...,DELETE_CANDIDATE,junk_filename,201728,2026-03-02 06:15:04.759999990,.db,NaN,NaN,NaN,NaN
36,Z:\SONADO-IK-801455240\10_PROJECTS\HTL0049-01_...,DELETE_CANDIDATE,junk_extension;duplicate_nonkeeper,664636,2022-11-02 15:31:59.000000000,.bak,055c707cbb87860279fe4c5d0c54e25f469745f4667040...,68e798b735d59dc43921dfd2f747dee588f7c8089ba6e8...,100.0,False
37,Z:\SONADO-IK-801455240\10_PROJECTS\HTL0049-01_...,DELETE_CANDIDATE,duplicate_nonkeeper,692091,2022-11-02 15:31:59.000000000,.dwg,e807257c06127f08c623938fb92bd9912d64e8aea54406...,98680a7abb133af8e4a5982793a7fd3d755f8ddab3b38a...,103.0,False
38,Z:\SONADO-IK-801455240\10_PROJECTS\HTL0049-01_...,DELETE_CANDIDATE,junk_extension;duplicate_nonkeeper,9554920,2022-11-02 15:31:59.000000000,.bak,fa7831c4357200d18b7a05593acf8267f6dd61e09aec30...,c5f40c6bcf021969264cd1a41f9649d621196426e3fd76...,332.0,False
39,Z:\SONADO-IK-801455240\10_PROJECTS\HTL0049-01_...,DELETE_CANDIDATE,duplicate_nonkeeper,9598848,2022-11-02 15:31:59.000000000,.dwg,038f8df44a1ab0393616d561ebba2d31887edd94d2a899...,26ea00678608e44154dc5aa8808566c719c4837a6bd1d7...,333.0,False
40,Z:\SONADO-IK-801455240\10_PROJECTS\HTL0049-01_...,DELETE_CANDIDATE,junk_extension;duplicate_nonkeeper,26347123,2022-11-02 15:31:59.000000000,.bak,4bd5483f21da657324d6e1bc37eae60ca8317525786854...,e4403d8df4dc956d35e48fe244ee0e9f9d0b1faec6b80e...,347.0,False
41,Z:\SONADO-IK-801455240\10_PROJECTS\HTL0049-01_...,DELETE_CANDIDATE,duplicate_nonkeeper,25864399,2022-11-02 15:31:59.000000000,.dwg,39f3f0e383d75266718cc953ea381f7f3549366440f084...,6ad4e54ec9e9255e0831849cebf97260796907108cbde3...,346.0,False
42,Z:\SONADO-IK-801455240\10_PROJECTS\HTL0049-01_...,DELETE_CANDIDATE,junk_extension;duplicate_nonkeeper,14532003,2022-11-02 15:31:59.000000000,.bak,6cbf074e5d49b8b24b1c6201390416d929bcfe5ed02596...,f6689a35b8ddf1cb3c55d13adcfab6394b1f5ea57786b5...,337.0,False
43,Z:\SONADO-IK-801455240\10_PROJECTS\HTL0049-01_...,DELETE_CANDIDATE,duplicate_nonkeeper,14532198,2022-11-02 15:31:59.000000000,.dwg,22f771bf30cd1ced37a60ee949b7a4c52953f822d9ad7e...,292b5fd7480f9e9734b035eaec274b6b4daa2a0cd5706d...,338.0,False
47,Z:\SONADO-IK-801455240\10_PROJECTS\HTL0049-01_...,DELETE_CANDIDATE,junk_extension;duplicate_keeper,664636,2022-11-02 15:31:59.000000000,.bak,055c707cbb87860279fe4c5d0c54e25f469745f4667040...,68e798b735d59dc43921dfd2f747dee588f7c8089ba6e8...,100.0,True


In [9]:
# Counts by reason (for DELETE_CANDIDATE rows)
tmp = out[out['decision'] == 'DELETE_CANDIDATE'].copy()
tmp['reason_list'] = tmp['reason'].fillna('').apply(lambda s: [r for r in s.split(';') if r])
reason_counts = (
    tmp.explode('reason_list')
       .query("reason_list != ''")
       .groupby('reason_list')
       .size()
       .sort_values(ascending=False)
)
reason_counts.head(50)


reason_list
duplicate_nonkeeper    503
junk_extension          39
junk_filename           20
duplicate_keeper        11
dtype: int64

In [10]:
# Largest delete candidates (often duplicates)
out[out["decision"] == "DELETE_CANDIDATE"].sort_values("size", ascending=False).head(50)

,path,decision,reason,size,mtime_iso,ext,hash_fast,hash_full,dup_group_id,dup_is_keeper
974,Z:\SONADO-IK-801455240\10_PROJECTS\HTL0049-01_...,DELETE_CANDIDATE,junk_extension;duplicate_nonkeeper,28062159,2022-11-02 15:31:59.000000000,.bak,0c9c733d7b975278214c02ab89567faca1f5960e1f46f4...,5b3727d7e1f6b61acacf45d44f5fee7cb246d4538add23...,350.0,False
55,Z:\SONADO-IK-801455240\10_PROJECTS\HTL0049-01_...,DELETE_CANDIDATE,junk_extension;duplicate_keeper,28062159,2022-11-02 15:31:59.000000000,.bak,0c9c733d7b975278214c02ab89567faca1f5960e1f46f4...,5b3727d7e1f6b61acacf45d44f5fee7cb246d4538add23...,350.0,True
650,Z:\SONADO-IK-801455240\10_PROJECTS\HTL0049-01_...,DELETE_CANDIDATE,junk_extension;duplicate_nonkeeper,28062159,2022-11-02 15:31:59.000000000,.bak,0c9c733d7b975278214c02ab89567faca1f5960e1f46f4...,5b3727d7e1f6b61acacf45d44f5fee7cb246d4538add23...,350.0,False
975,Z:\SONADO-IK-801455240\10_PROJECTS\HTL0049-01_...,DELETE_CANDIDATE,duplicate_nonkeeper,27783962,2022-11-02 15:31:59.000000000,.dwg,896982e43dd1ec115dac5669a29362d5c5f8721c97af75...,d98a7ac2f56dabc05853f4f124949754d88626f3a11547...,349.0,False
651,Z:\SONADO-IK-801455240\10_PROJECTS\HTL0049-01_...,DELETE_CANDIDATE,duplicate_nonkeeper,27783962,2022-11-02 15:31:59.000000000,.dwg,896982e43dd1ec115dac5669a29362d5c5f8721c97af75...,d98a7ac2f56dabc05853f4f124949754d88626f3a11547...,349.0,False
680,Z:\SONADO-IK-801455240\10_PROJECTS\HTL0049-01_...,DELETE_CANDIDATE,duplicate_nonkeeper,27562520,2022-11-02 15:31:59.000000000,.pdf,2b7d6ab5499c268b2bc64b1d150241132091dce3098032...,bb898e194bb2a0b83ddf8d7ad6bbffc4eb0f2e559f0894...,348.0,False
1005,Z:\SONADO-IK-801455240\10_PROJECTS\HTL0049-01_...,DELETE_CANDIDATE,duplicate_nonkeeper,27562520,2022-11-02 15:31:59.000000000,.pdf,2b7d6ab5499c268b2bc64b1d150241132091dce3098032...,bb898e194bb2a0b83ddf8d7ad6bbffc4eb0f2e559f0894...,348.0,False
646,Z:\SONADO-IK-801455240\10_PROJECTS\HTL0049-01_...,DELETE_CANDIDATE,junk_extension;duplicate_nonkeeper,26347123,2022-11-02 15:31:59.000000000,.bak,4bd5483f21da657324d6e1bc37eae60ca8317525786854...,e4403d8df4dc956d35e48fe244ee0e9f9d0b1faec6b80e...,347.0,False
40,Z:\SONADO-IK-801455240\10_PROJECTS\HTL0049-01_...,DELETE_CANDIDATE,junk_extension;duplicate_nonkeeper,26347123,2022-11-02 15:31:59.000000000,.bak,4bd5483f21da657324d6e1bc37eae60ca8317525786854...,e4403d8df4dc956d35e48fe244ee0e9f9d0b1faec6b80e...,347.0,False
51,Z:\SONADO-IK-801455240\10_PROJECTS\HTL0049-01_...,DELETE_CANDIDATE,junk_extension;duplicate_keeper,26347123,2022-11-02 15:31:59.000000000,.bak,4bd5483f21da657324d6e1bc37eae60ca8317525786854...,e4403d8df4dc956d35e48fe244ee0e9f9d0b1faec6b80e...,347.0,True


## 8) OPTIONAL: Quarantine move (safer than delete)
**Default is disabled.** If you enable it, files are moved to a quarantine folder preserving relative paths.

⚠️ Only do this after reviewing the CSV.


In [13]:
ENABLE_QUARANTINE = True
QUARANTINE_DIR = Path("..") / "quarantine" / f"run_{RUN_TAG}"

def move_to_quarantine(src_path: str, root: Path, quarantine_root: Path):
    src = Path(src_path)
    rel = src.relative_to(root)
    dst = quarantine_root / rel
    dst.parent.mkdir(parents=True, exist_ok=True)

    if dst.exists():
        raise FileExistsError(f"Destination already exists: {dst}")

    try:
        src.replace(dst)  # atomic move on same volume
    except OSError:
        # Cross-volume fallback (e.g., Z: -> C:)
        import shutil
        shutil.move(str(src), str(dst))

if ENABLE_QUARANTINE:
    QUARANTINE_DIR.mkdir(parents=True, exist_ok=True)
    to_move = out[out["decision"] == "DELETE_CANDIDATE"]["path"].tolist()
    moved = 0
    for p in tqdm(to_move, desc="Quarantining"):
        try:
            move_to_quarantine(p, ROOT, QUARANTINE_DIR)
            moved += 1
        except Exception as e:
            print("FAILED:", p, "->", e)
    print(f"Moved {moved}/{len(to_move)} files to {QUARANTINE_DIR}")
else:
    print("Quarantine is disabled (ENABLE_QUARANTINE=False).")

Quarantining: 100%|██████████| 527/527 [10:44<00:00,  1.22s/it]

Moved 527/527 files to ..\quarantine\run_20260302_141826


In [9]:
def delete_empty_folders(root_dir: Path | str, *, remove_root: bool = False, dry_run: bool = True) -> int:
    """Delete empty folders under root_dir (bottom-up).
    
    Args:
        root_dir: Directory to clean.
        remove_root: Also remove root_dir if it ends up empty.
        dry_run: If True, only prints what would be deleted.
    
    Returns:
        Number of directories deleted (or that would be deleted in dry_run mode).
    """
    root = Path(root_dir)
    if not root.exists() or not root.is_dir():
        raise ValueError(f"Not a valid directory: {root}")

    deleted_count = 0
    for current, dirs, files in os.walk(root, topdown=False):
        current_path = Path(current)
        if files or dirs:
            continue
        if current_path == root and not remove_root:
            continue

        try:
            if dry_run:
                print(f"[DRY RUN] Would delete: {current_path}")
            else:
                current_path.rmdir()
            deleted_count += 1
        except OSError:
            # Directory may have changed or no longer be empty; skip safely
            pass

    mode = "would be deleted" if dry_run else "deleted"
    print(f"Empty folders {mode}: {deleted_count}")
    return deleted_count

# Example usage:
# delete_empty_folders(ROOT, dry_run=True)
delete_empty_folders(ROOT, dry_run=False)

Empty folders deleted: 11


11

## 9) Document quarantine candidates CSV
Create a dedicated CSV from the *current remaining files* and flag document-like files for quarantine.

- CAD / drawing formats are kept
- Document files are flagged as `QUARANTINE_DOCUMENT`
- PDF handling is heuristic: PDFs with drawing-related keywords in path/name are treated as drawing PDFs and kept

In [6]:
# Build a fresh metadata table from what currently remains under ROOT
import re
import fitz  # PyMuPDF

remaining_rows = []
for rec in tqdm(iter_files(ROOT), desc="Scanning remaining", unit="files"):
    remaining_rows.append(rec)

remaining_df = pd.DataFrame(remaining_rows)

if remaining_df.empty:
    print("No files found under ROOT.")
else:
    remaining_df["mtime_iso"] = pd.to_datetime(
        remaining_df["mtime"], unit="s", utc=True
    ).dt.tz_convert(None).astype(str)

    CAD_EXTENSIONS = {
        ".dwg", ".dxf", ".dwt", ".dwf", ".dgn", ".rvt", ".ifc", ".skp"
    }
    DOCUMENT_EXTENSIONS = {
        ".doc", ".docx", ".txt", ".rtf", ".odt"
    }

    DOC_KEYWORDS_IN_TEXT = [
        "declaration", "assignment", "technical", "description", "schedule", "method",
        "scope", "requirements", "analysis", "calculation", "conclusion", "report",
        "fem", "lab", "specification", "chronik", "programmat",
        "δήλωση", "ανάθεση", "τεχνική", "περιγραφή", "χρονικ", "προγραμματισ",
        "μελέτη", "ανάλυση", "έκθεση", "προδιαγραφ"
    ]
    DRAWING_KEYWORDS_IN_TEXT = [
        "scale", "legend", "grid", "detail", "section", "elevation", "plan view",
        "title block", "sheet", "revision",
        "κάτοψη", "τομή", "όψη", "κλίμακα", "λεπτομέρεια"
    ]

    def _count_hits(text: str, keywords: list[str]) -> int:
        low = text.lower()
        return sum(1 for kw in keywords if kw in low)

    def classify_pdf_with_pymupdf(path_str: str) -> tuple[str, str, str, dict]:
        """
        Returns: (file_category, decision, reason, metrics)
        file_category in {PDF_TEXT_DOCUMENT, PDF_RASTER_IMAGE, PDF_VECTOR_DRAWING}
        """
        p = Path(path_str)

        total_words = 0
        doc_kw_hits = 0
        drawing_kw_hits = 0
        text_pages = 0
        image_pages = 0
        total_images = 0
        total_drawings = 0
        total_page_area = 0.0
        total_image_block_area = 0.0

        try:
            pdf = fitz.open(str(p))
        except Exception:
            return "PDF_TEXT_DOCUMENT", "QUARANTINE_DOCUMENT", "pdf_open_failed_default_document", {
                "word_count": 0,
                "doc_kw_hits": 0,
                "drawing_kw_hits": 0,
                "image_ratio": 0.0,
                "drawing_count": 0,
            }

        max_pages = min(len(pdf), 60)
        for page_idx in range(max_pages):
            page = pdf[page_idx]
            page_text = page.get_text("text") or ""
            words = re.findall(r"[A-Za-zΑ-Ωα-ω]{3,}", page_text)
            word_count_page = len(words)
            total_words += word_count_page
            if word_count_page >= 20:
                text_pages += 1

            doc_kw_hits += _count_hits(page_text, DOC_KEYWORDS_IN_TEXT)
            drawing_kw_hits += _count_hits(page_text, DRAWING_KEYWORDS_IN_TEXT)

            drawings = page.get_drawings()
            total_drawings += len(drawings)

            imgs = page.get_images(full=False)
            total_images += len(imgs)
            if len(imgs) > 0:
                image_pages += 1

            page_area = float(page.rect.width * page.rect.height)
            total_page_area += page_area

            try:
                text_dict = page.get_text("dict")
                for block in text_dict.get("blocks", []):
                    if block.get("type") == 1:  # image block
                        x0, y0, x1, y1 = block.get("bbox", (0, 0, 0, 0))
                        total_image_block_area += max(0.0, (x1 - x0) * (y1 - y0))
            except Exception:
                pass

        pdf.close()

        image_ratio = (total_image_block_area / total_page_area) if total_page_area > 0 else 0.0

        doc_score = (total_words * 1.8) + (doc_kw_hits * 50) + (text_pages * 25)
        drawing_score = (total_drawings * 2.5) + (drawing_kw_hits * 45)
        raster_score = (image_ratio * 180) + (image_pages * 20) + (total_images * 0.8)

        metrics = {
            "word_count": int(total_words),
            "doc_kw_hits": int(doc_kw_hits),
            "drawing_kw_hits": int(drawing_kw_hits),
            "text_pages": int(text_pages),
            "image_pages": int(image_pages),
            "drawing_count": int(total_drawings),
            "image_count": int(total_images),
            "image_ratio": round(float(image_ratio), 4),
            "doc_score": round(float(doc_score), 2),
            "drawing_score": round(float(drawing_score), 2),
            "raster_score": round(float(raster_score), 2),
        }

        # Strong document signals (Greek or English text-heavy content)
        if total_words >= 180:
            return "PDF_TEXT_DOCUMENT", "QUARANTINE_DOCUMENT", "pdf_text_wordcount", metrics
        if doc_kw_hits >= max(2, drawing_kw_hits):
            return "PDF_TEXT_DOCUMENT", "QUARANTINE_DOCUMENT", "pdf_text_keywords", metrics

        # Strong drawing/raster signals
        if drawing_score >= max(doc_score * 1.35, raster_score * 1.1) and total_drawings >= 40:
            return "PDF_VECTOR_DRAWING", "KEEP", "pdf_vector_drawing", metrics

        if raster_score >= max(doc_score * 1.35, drawing_score * 1.1) and image_ratio >= 0.35:
            return "PDF_RASTER_IMAGE", "KEEP", "pdf_raster_image", metrics

        # Unclear between drawing/document -> quarantine for safer review
        if doc_score >= drawing_score * 0.9:
            return "PDF_TEXT_DOCUMENT", "QUARANTINE_DOCUMENT", "pdf_unclear_default_document", metrics

        # If clearly non-document profile
        if raster_score >= drawing_score:
            return "PDF_RASTER_IMAGE", "KEEP", "pdf_unclear_bias_raster", metrics
        return "PDF_VECTOR_DRAWING", "KEEP", "pdf_unclear_bias_drawing", metrics

    def classify_row(path_str: str, ext: str) -> tuple[str, str, str, dict]:
        ext_l = (ext or "").lower()

        if ext_l in CAD_EXTENSIONS:
            return "DRAWING_CAD", "KEEP", "cad_format", {}

        if ext_l == ".pdf":
            return classify_pdf_with_pymupdf(path_str)

        if ext_l in DOCUMENT_EXTENSIONS:
            return "DOCUMENT", "QUARANTINE_DOCUMENT", "document_format", {}

        return "OTHER", "KEEP", "not_document_target", {}

    classified = remaining_df.apply(
        lambda r: classify_row(r["path"], r["ext"]),
        axis=1,
        result_type="expand"
    )
    classified.columns = ["file_category", "decision", "reason", "pdf_metrics"]
    remaining_df = pd.concat([remaining_df, classified], axis=1)

    # Flatten selected PDF metrics for auditability in CSV
    def metric_get(d: dict, key: str):
        if isinstance(d, dict):
            return d.get(key)
        return None

    remaining_df["pdf_word_count"] = remaining_df["pdf_metrics"].apply(lambda d: metric_get(d, "word_count"))
    remaining_df["pdf_doc_kw_hits"] = remaining_df["pdf_metrics"].apply(lambda d: metric_get(d, "doc_kw_hits"))
    remaining_df["pdf_drawing_kw_hits"] = remaining_df["pdf_metrics"].apply(lambda d: metric_get(d, "drawing_kw_hits"))
    remaining_df["pdf_drawing_count"] = remaining_df["pdf_metrics"].apply(lambda d: metric_get(d, "drawing_count"))
    remaining_df["pdf_image_ratio"] = remaining_df["pdf_metrics"].apply(lambda d: metric_get(d, "image_ratio"))

    DOC_CSV_PATH = OUTPUT_DIR / f"document_quarantine_candidates_{RUN_TAG}.csv"
    doc_out = remaining_df[[
        "path", "name", "ext", "size", "mtime_iso",
        "file_category", "decision", "reason",
        "pdf_word_count", "pdf_doc_kw_hits", "pdf_drawing_kw_hits", "pdf_drawing_count", "pdf_image_ratio"
    ]].copy()

    doc_out.to_csv(DOC_CSV_PATH, index=False, encoding="utf-8")

    print("Wrote:", DOC_CSV_PATH)
    print("\nDecision counts:")
    print(doc_out["decision"].value_counts(dropna=False))
    print("\nCategory counts:")
    print(doc_out["file_category"].value_counts(dropna=False))

    doc_out[doc_out["decision"] == "QUARANTINE_DOCUMENT"].head(50)

Scanning remaining: 440files [01:44,  4.23files/s]


MuPDF error: format error: No default Layer config

MuPDF error: format error: No default Layer config

MuPDF error: format error: No default Layer config

MuPDF error: format error: No default Layer config

MuPDF error: format error: No default Layer config

MuPDF error: format error: No default Layer config

MuPDF error: format error: No default Layer config

MuPDF error: format error: No default Layer config

MuPDF error: format error: No default Layer config

MuPDF error: format error: No default Layer config

MuPDF error: format error: No default Layer config

MuPDF error: format error: No default Layer config

MuPDF error: format error: No default Layer config

MuPDF error: format error: No default Layer config

MuPDF error: format error: No default Layer config

MuPDF error: format error: No default Layer config

MuPDF error: format error: No default Layer config

MuPDF error: format error: No default Layer config

MuPDF error: format error: No default Layer config

MuPDF error:

In [8]:
# Quarantine only PDF text documents and non-PDF documents from doc_out
DOC_ONLY_ENABLE_QUARANTINE = True
DOC_ONLY_DRY_RUN = False
DOC_ONLY_QUARANTINE_DIR = Path("..") / "quarantine" / f"run_{RUN_TAG}" / "documents_only"

if "doc_out" not in globals():
    raise RuntimeError("Run Cell 24 first to build doc_out.")

# target only requested categories
doc_targets = doc_out[
    doc_out["file_category"].isin(["PDF_TEXT_DOCUMENT", "DOCUMENT"])
].copy()

# keep safeguard aligned with decision flag
if "decision" in doc_targets.columns:
    doc_targets = doc_targets[doc_targets["decision"] == "QUARANTINE_DOCUMENT"].copy()

print(f"Document quarantine candidates: {len(doc_targets)}")


def move_to_doc_quarantine(src_path: str, root: Path, quarantine_root: Path):
    src = Path(src_path)
    rel = src.relative_to(root)
    dst = quarantine_root / rel
    dst.parent.mkdir(parents=True, exist_ok=True)

    if dst.exists():
        raise FileExistsError(f"Destination already exists: {dst}")

    try:
        src.replace(dst)  # same-volume fast path
    except OSError:
        import shutil
        shutil.move(str(src), str(dst))


if DOC_ONLY_ENABLE_QUARANTINE:
    DOC_ONLY_QUARANTINE_DIR.mkdir(parents=True, exist_ok=True)
    moved = 0
    failed = 0

    for p in tqdm(doc_targets["path"].tolist(), desc="Quarantining docs"):
        try:
            if DOC_ONLY_DRY_RUN:
                print(f"[DRY RUN] Would move: {p}")
            else:
                move_to_doc_quarantine(p, ROOT, DOC_ONLY_QUARANTINE_DIR)
            moved += 1
        except Exception as e:
            failed += 1
            print("FAILED:", p, "->", e)

    action = "Would move" if DOC_ONLY_DRY_RUN else "Moved"
    print(f"{action} {moved}/{len(doc_targets)} files to {DOC_ONLY_QUARANTINE_DIR} (failed: {failed})")
else:
    print("Document-only quarantine disabled.")

Document quarantine candidates: 124


Quarantining docs: 100%|██████████| 124/124 [02:40<00:00,  1.30s/it]

Moved 124/124 files to ..\quarantine\run_20260303_081557\documents_only (failed: 0)


In [4]:
import csv
import os
import re
import unicodedata
from datetime import datetime
from pathlib import Path

GREEK_TO_LATIN = {
    "α": "a", "β": "v", "γ": "g", "δ": "d", "ε": "e", "ζ": "z", "η": "i", "θ": "th",
    "ι": "i", "κ": "k", "λ": "l", "μ": "m", "ν": "n", "ξ": "x", "ο": "o", "π": "p",
    "ρ": "r", "σ": "s", "ς": "s", "τ": "t", "υ": "y", "φ": "f", "χ": "ch", "ψ": "ps",
    "ω": "o",
}


def to_greeklish(value: str) -> str:
    normalized = unicodedata.normalize("NFD", value)
    no_accents = "".join(ch for ch in normalized if unicodedata.category(ch) != "Mn")
    lowered = no_accents.lower()
    return "".join(GREEK_TO_LATIN.get(ch, ch) for ch in lowered)


def normalize_file_stem(stem: str) -> str:
    greeklish = to_greeklish(stem)
    greeklish = re.sub(r"[\s_]+", "-", greeklish)
    greeklish = re.sub(r"[^a-z0-9-]", "", greeklish)
    greeklish = re.sub(r"-+", "-", greeklish).strip("-")
    return greeklish or "file"


def build_target_name(file_path: Path, prefix: str) -> str:
    created_on = datetime.fromtimestamp(file_path.stat().st_ctime).strftime("%Y%m%d")
    cleaned_stem = normalize_file_stem(file_path.stem)
    extension = file_path.suffix.lower()
    return f"{prefix}{cleaned_stem}_{created_on}_v01_FINAL{extension}"


def is_already_standardized(file_path: Path, prefix: str) -> bool:
    stem = file_path.stem
    return stem.startswith(prefix) and stem.endswith("_v01_FINAL")


def export_rename_log(log_rows: list[dict], log_csv_path: str | Path | None = None) -> Path:
    if log_csv_path is None:
        out_dir = Path("..") / "outputs"
        out_dir.mkdir(parents=True, exist_ok=True)
        ts = datetime.now().strftime("%Y%m%d_%H%M%S")
        log_path = out_dir / f"rename_log_{ts}.csv"
    else:
        log_path = Path(log_csv_path)
        log_path.parent.mkdir(parents=True, exist_ok=True)

    fieldnames = ["source_path", "target_path", "status", "error", "logged_at"]
    with log_path.open("w", newline="", encoding="utf-8-sig") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(log_rows)

    return log_path


def rename_files_non_underscore_dirs(
    root_folder: str,
    prefix: str = "HTL00p049-01_CN_DRWTEC_",
    dry_run: bool = True,
    log_csv_path: str | Path | None = None,
):
    root = Path(root_folder)
    if not root.exists() or not root.is_dir():
        raise FileNotFoundError(f"Folder not found: {root}")

    planned_ops = []

    for current_dir, dirs, files in os.walk(root):
        dirs[:] = [directory for directory in dirs if not directory.startswith("_")]

        current_path = Path(current_dir)
        if current_path.name.startswith("_"):
            continue

        for filename in files:
            source = current_path / filename

            if is_already_standardized(source, prefix):
                continue

            target_name = build_target_name(source, prefix)

            if source.name == target_name:
                continue

            target = source.with_name(target_name)

            if target.exists() and target != source:
                counter = 2
                while True:
                    candidate_target = source.with_name(
                        f"{target.stem}-{counter:02d}{target.suffix.lower()}"
                    )
                    if not candidate_target.exists():
                        target = candidate_target
                        break
                    counter += 1

            planned_ops.append((source, target))

    if dry_run:
        print(f"[DRY-RUN] Files to rename: {len(planned_ops)}")
        for src, dst in planned_ops[:20]:
            print(f"{src.name}  ->  {dst.name}")
        if len(planned_ops) > 20:
            print(f"... and {len(planned_ops) - 20} more")

        dry_rows = [
            {
                "source_path": str(src),
                "target_path": str(dst),
                "status": "PLANNED",
                "error": "",
                "logged_at": datetime.now().isoformat(timespec="seconds"),
            }
            for src, dst in planned_ops
        ]
        log_path = export_rename_log(dry_rows, log_csv_path=log_csv_path)
        print(f"CSV log written: {log_path}")
        return planned_ops

    run_rows = []
    renamed_count = 0

    for src, dst in planned_ops:
        try:
            src.rename(dst)
            renamed_count += 1
            run_rows.append(
                {
                    "source_path": str(src),
                    "target_path": str(dst),
                    "status": "RENAMED",
                    "error": "",
                    "logged_at": datetime.now().isoformat(timespec="seconds"),
                }
            )
        except Exception as exc:
            run_rows.append(
                {
                    "source_path": str(src),
                    "target_path": str(dst),
                    "status": "FAILED",
                    "error": str(exc),
                    "logged_at": datetime.now().isoformat(timespec="seconds"),
                }
            )

    log_path = export_rename_log(run_rows, log_csv_path=log_csv_path)
    print(f"Renamed {renamed_count}/{len(planned_ops)} files.")
    print(f"CSV log written: {log_path}")
    return planned_ops


ROOT_PATH = r"Z:\SONADO-IK-801455240\10_PROJECTS\HTL0049-01_OITYLO-KOKKALA_MANI\04_DESIGN_ENGINEERING"
RENAME_LOG_PATH = (Path("..") / "outputs" / f"rename_log_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv")

# 1) Preview changes first (+ CSV log)
preview = rename_files_non_underscore_dirs(ROOT_PATH, dry_run=True, log_csv_path=RENAME_LOG_PATH)

# 2) Uncomment to perform actual rename (+ CSV log)
# rename_files_non_underscore_dirs(ROOT_PATH, dry_run=False, log_csv_path=RENAME_LOG_PATH)


[DRY-RUN] Files to rename: 0
CSV log written: ..\outputs\rename_log_20260305_082723.csv


# Preproduction

In [39]:
from datetime import datetime
from pathlib import Path
import importlib
import sys

# Ensure src package is importable from notebooks/
workspace_root = Path.cwd().resolve().parent
src_path = workspace_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

import file_server_cleanup.inventory as inventory_module
inventory_module = importlib.reload(inventory_module)

inventory_csv = workspace_root / "outputs" / f"file_inventory_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
# scan_root = Path(ROOT_PATH) if "ROOT_PATH" in globals() else Path(r"Z:\SONADO-IK-801455240\10_PROJECTS\HTL0049-01_OITYLO-KOKKALA_MANI\04_DESIGN_ENGINEERING")
scan_root = Path(r"C:\Users\User\Desktop\SONADO Μ.ΙΚΕ-ΚΟΚΚΑΛΑ ΟΙΤΥΛΟ (N.4399)")

# Allowed extension list for extension flagging
ALLOWED_EXTENSIONS = {
    ".dwg", ".dxf", ".dwt", ".dwf", ".dgn", ".rvt", ".ifc", ".skp",
    ".pdf", ".doc", ".docx", ".txt", ".rtf", ".odt",
    ".xls", ".xlsx", ".csv",
    ".jpg", ".jpeg", ".png", ".tif", ".tiff",
}

result = inventory_module.export_file_inventory_csv(
    root_path=scan_root,
    output_csv_path=inventory_csv,
    skip_hidden_dirs=False,
    compute_content_duplicates=False,  # placeholder mode
    allowed_extensions=ALLOWED_EXTENSIONS,
)

print("CSV:", result.csv_path)
print("Scanned files:", result.scanned_files)
print("Duplicate by name+ext+created_date:", result.duplicate_name_ext_created_files)
print("Duplicate by content:", result.duplicate_content_files)
print(f"Unique file types found ({len(result.unique_file_types)}):", list(result.unique_file_types))


CSV: C:\00_dev\file_server_clean_up\outputs\file_inventory_20260305_153519.csv
Scanned files: 3982
Duplicate by name+ext+created_date: 2565
Duplicate by content: 0
Unique file types found (23): ['.7z', '.bak', '.db', '.doc', '.docx', '.dwg', '.dwl', '.dwl2', '.eml', '.jpeg', '.jpg', '.lnk', '.msg', '.odt', '.pdf', '.png', '.rar', '.tfw', '.tif', '.txt', '.xls', '.xlsx', '.zip']


In [41]:
import importlib
import file_server_cleanup.inventory as inventory_module
inventory_module = importlib.reload(inventory_module)

if "scan_root" not in globals() or "inventory_csv" not in globals():
    raise RuntimeError("Run Cell 28 first to generate inventory_csv and scan_root.")

move_result = inventory_module.move_duplicated_files_from_csv(
    scan_root=scan_root,
    inventory_csv_path=inventory_csv,
    duplicated_folder_name="_DUPLICATED",
    dry_run=True,
)

print("CSV:", move_result.csv_path)
print("Duplicated folder:", move_result.duplicated_folder)
print("Duplicate rows in CSV:", move_result.duplicate_rows)
print("Planned moves:", move_result.planned_moves)
print("Moved files:", move_result.moved_files)
print("Skipped files:", move_result.skipped_files)
print("Failed files:", move_result.failed_files)

# Uncomment to perform the move:
move_result = inventory_module.move_duplicated_files_from_csv(
    scan_root=scan_root,
    inventory_csv_path=inventory_csv,
    duplicated_folder_name="_DUPLICATED",
    dry_run=False,
)


CSV: C:\00_dev\file_server_clean_up\outputs\file_inventory_20260305_153519.csv
Duplicated folder: C:\Users\User\Desktop\SONADO Μ.ΙΚΕ-ΚΟΚΚΑΛΑ ΟΙΤΥΛΟ (N.4399)\_DUPLICATED
Duplicate rows in CSV: 2565
Planned moves: 1682
Moved files: 0
Skipped files: 883
Failed files: 0


In [49]:
import importlib
import file_server_cleanup.inventory as inventory_module
inventory_module = importlib.reload(inventory_module)

if "scan_root" not in globals() or "inventory_csv" not in globals():
    raise RuntimeError("Run Cell 28 first to generate inventory_csv and scan_root.")

deprecated_result = inventory_module.move_deprecated_files_from_csv(
    scan_root=scan_root,
    inventory_csv_path=inventory_csv,
    deprecated_folder_name="_DEPRECATED",
    dry_run=True,
)

print("CSV:", deprecated_result.csv_path)
print("Deprecated folder:", deprecated_result.deprecated_folder)
print("Deprecated rows in CSV:", deprecated_result.deprecated_rows)
print("Planned moves:", deprecated_result.planned_moves)
print("Moved files:", deprecated_result.moved_files)
print("Skipped files:", deprecated_result.skipped_files)
print("Failed files:", deprecated_result.failed_files)

# Uncomment to perform the move:
deprecated_result = inventory_module.move_deprecated_files_from_csv(
    scan_root=scan_root,
    inventory_csv_path=inventory_csv,
    deprecated_folder_name="_DEPRECATED",
    dry_run=False,
)


CSV: C:\00_dev\file_server_clean_up\outputs\file_inventory_20260305_153519.csv
Deprecated folder: C:\Users\User\Desktop\SONADO Μ.ΙΚΕ-ΚΟΚΚΑΛΑ ΟΙΤΥΛΟ (N.4399)\_DEPRECATED
Deprecated rows in CSV: 353
Planned moves: 76
Moved files: 0
Skipped files: 277
Failed files: 0


In [51]:
import importlib
import file_server_cleanup.inventory as inventory_module
inventory_module = importlib.reload(inventory_module)

if "scan_root" not in globals():
    raise RuntimeError("Run Cell 28 first to set scan_root.")

EMPTY_DIR_DRY_RUN = True
DELETE_EMPTY_DIRS = True

empty_dirs_result = inventory_module.delete_empty_folders(
    root_path=scan_root,
    remove_root=False,
    dry_run=EMPTY_DIR_DRY_RUN,
    exclude_dir_names={"_DUPLICATED", "_DEPRECATED"},
)

mode = "DRY RUN" if empty_dirs_result.dry_run else "EXECUTE"
print(f"Mode: {mode}")
print("Root:", empty_dirs_result.root_path)
print("Inspected dirs:", empty_dirs_result.inspected_dirs)
print("Empty dirs to delete:" if empty_dirs_result.dry_run else "Deleted dirs:", empty_dirs_result.deleted_dirs)
print("Skipped dirs:", empty_dirs_result.skipped_dirs)
print("Failed dirs:", empty_dirs_result.failed_dirs)

if DELETE_EMPTY_DIRS:
    empty_dirs_execute = inventory_module.delete_empty_folders(
        root_path=scan_root,
        remove_root=False,
        dry_run=False,
        exclude_dir_names={"_DUPLICATED", "_DEPRECATED"},
    )
    print("\nExecution complete")
    print("Deleted dirs:", empty_dirs_execute.deleted_dirs)
    print("Skipped dirs:", empty_dirs_execute.skipped_dirs)
    print("Failed dirs:", empty_dirs_execute.failed_dirs)
else:
    print("\nSet DELETE_EMPTY_DIRS = True to perform actual deletion.")

Mode: DRY RUN
Root: C:\Users\User\Desktop\SONADO Μ.ΙΚΕ-ΚΟΚΚΑΛΑ ΟΙΤΥΛΟ (N.4399)
Inspected dirs: 884
Empty dirs to delete: 0
Skipped dirs: 3
Failed dirs: 0

Execution complete
Deleted dirs: 0
Skipped dirs: 3
Failed dirs: 0
